# 03 — YOLOv8 Training
## SmartMine Vision AI · Stage 1: PPE Detection

---

### Objectives

1. Fine-tune **YOLOv8n** on the Construction Site Safety Dataset.
2. Use a reproducible training configuration.
3. Save the best model weights to `models/ppe/`.
4. Monitor training curves (loss, mAP) during training.

---

### Why YOLOv8n?

| Variant | Parameters | Speed | mAP COCO |
|---------|-----------|-------|----------|
| YOLOv8n | 3.2M | Fastest | 37.3 |
| YOLOv8s | 11.2M | Fast | 44.9 |
| YOLOv8m | 25.9M | Moderate | 50.2 |

We start with **YOLOv8n** (nano) for fast iteration. After evaluation, we may upgrade to YOLOv8s if accuracy is insufficient.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

_nb_dir = Path().resolve()
PROJECT_ROOT = _nb_dir
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.ppe_detection.utils import CONFIGS_DIR, MODELS_DIR, EXPERIMENTS_DIR, ensure_dirs
from src.ppe_detection.trainer import train_ppe_model

ensure_dirs()

DATA_YAML = CONFIGS_DIR / "ppe_dataset.yaml"
print(f"Config: {DATA_YAML}")
print(f"Models dir: {MODELS_DIR}")
print(f"Experiments dir: {EXPERIMENTS_DIR}")

## 2. Training Configuration

| Parameter | Value | Rationale |
|-----------|-------|----------|
| Base model | yolov8n.pt | Fast iteration baseline |
| Epochs | 100 | Sufficient for fine-tuning |
| Image size | 640 | Standard YOLO input (matches dataset pre-processing) |
| Batch | -1 (auto) | YOLO auto-selects optimal batch for available VRAM |
| Optimizer | Auto (AdamW) | Ultralytics default — proven effective |
| Device | 0 (GPU) | Change to `cpu` if no GPU available |

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = "0"
else:
    print("Training on CPU — will be slower.")
    device = "cpu"

## 3. Launch Training

> **Expected time:**  
> GPU (RTX 3060+): ~20–40 min for 100 epochs  
> CPU only: several hours — reduce epochs to 20 for a quick test run

Training logs and weights are saved automatically to `experiments/ppe_v1/baseline/`.

In [ ]:
best_weights = train_ppe_model(
    data_yaml=DATA_YAML,
    base_model="yolov8n.pt",
    epochs=100,
    imgsz=640,
    name="baseline",
    device=device,
)
print(f"\nTraining complete. Best weights → {best_weights}")

## 4. Training Results

After training completes, Ultralytics saves these files automatically:

```
experiments/ppe_v1/baseline/
├── results.csv          ← per-epoch metrics
├── results.png          ← loss and mAP curves
├── confusion_matrix.png
├── weights/
│   ├── best.pt
│   └── last.pt
└── val_batch*.jpg
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

results_png = EXPERIMENTS_DIR / "ppe_v1" / "baseline" / "results.png"
if results_png.exists():
    img = mpimg.imread(str(results_png))
    plt.figure(figsize=(16, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Training Results")
    plt.show()
else:
    print(f"Not found: {results_png}")

## 5. Conclusions & Next Steps

**What to evaluate after training:**
- Is `mAP50` above 0.70? → Good baseline.
- Is `mAP50-95` above 0.45? → Acceptable.
- Are loss curves converging? → No overfitting sign.

**Next:** `04_evaluation.ipynb` — quantitative metrics, confusion matrix, precision-recall curves.